In [2]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import train_test_split
from torchvision import models
from torchvision import transforms

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

Using: cpu


In [6]:
X = np.load('../data/X.npy', mmap_mode='r')
y = np.load('../data/y.npy', mmap_mode='r')

In [7]:
print(X.shape, y.shape)

(28709, 224, 224, 3) (28709,)


In [8]:
print('Images are loaded as:', X.shape, '(N, H, W, C)')

Images are loaded as: (28709, 224, 224, 3) (N, H, W, C)


In [9]:
mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)

In [10]:
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
])

class EmotionDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        image = torch.from_numpy(np.array(self.images[idx], copy=True)).permute(2, 0, 1).float() / 255.0
        if self.transform is not None:
            image = self.transform(image)
        image = (image - mean) / std
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return image, label

indices = np.arange(len(y))
train_indices, val_indices = train_test_split(
    indices,
    test_size=0.2,
    random_state=42,
    stratify=np.asarray(y)
)

train_loader = DataLoader(Subset(EmotionDataset(X, y, transform=train_transform), train_indices), batch_size=32, shuffle=True)
val_loader = DataLoader(Subset(EmotionDataset(X, y), val_indices), batch_size=32)

In [11]:
print(f'Train batches: {len(train_loader)}, validation batches: {len(val_loader)}')

Train batches: 718, validation batches: 180


In [9]:
class EmotionCnn(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(3,32,3),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32,64,3),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.AdaptiveAvgPool2d((1,1))

        )

        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64,7)
        )

    def forward(self,x):
        x = self.conv(x)
        x = self.fc(x)
        return x

In [10]:
print(X.shape)

(28709, 224, 224, 3)


In [11]:
model = models.resnet18(pretrained=True)

model.fc = nn.Linear(model.fc.in_features, 7)

model = model.to(device)

for param in model.parameters():
    param.requires_grad = False

for param in model.layer4.parameters():
    param.requires_grad = True

for param in model.fc.parameters():
    param.requires_grad = True

loss_fn = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4
)

for epoch in range(15):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for i, (batch_X, batch_y) in enumerate(train_loader):
        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)
        preds = model(batch_X)
        loss = loss_fn(preds, batch_y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()*batch_X.size(0)

        _ , predicted = torch.max(preds, 1)
        correct += (predicted == batch_y).sum().item()
        total += batch_y.size(0)

        if i % 100 == 0:
            print(f"Epoch {epoch+1} | Batch {i}/{len(train_loader)} | Loss: {loss.item():.4f}")

    train_acc = correct/total
    
    #validation
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for batch_X, batch_y in val_loader:
            batch_X = batch_X.to(device)
            batch_y = batch_y.to(device)
            preds = model(batch_X)
            _ , predicted = torch.max(preds, 1)

            correct += (predicted == batch_y).sum().item()
            total += batch_y.size(0)

    val_acc = correct/total
    
    avg_loss = total_loss / len(train_loader.dataset)
    print(f"Epoch {epoch+1}, Loss: {avg_loss:.3f}, Train Acc: {train_acc:.3f} Val Acc: {val_acc:.3f}")

C:\Users\kusha\AppData\Roaming\Python\Python312\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\kusha\AppData\Roaming\Python\Python312\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Epoch 1 | Batch 0/718 | Loss: 2.0265
Epoch 1 | Batch 100/718 | Loss: 1.3663
Epoch 1 | Batch 200/718 | Loss: 1.7716
Epoch 1 | Batch 300/718 | Loss: 1.6056
Epoch 1 | Batch 400/718 | Loss: 1.6202
Epoch 1 | Batch 500/718 | Loss: 1.7235
Epoch 1 | Batch 600/718 | Loss: 1.6495
Epoch 1 | Batch 700/718 | Loss: 1.4304
Epoch 1, Loss: 1.509, Train Acc: 0.414 Val Acc: 0.466
Epoch 2 | Batch 0/718 | Loss: 1.4497
Epoch 2 | Batch 100/718 | Loss: 1.6572
Epoch 2 | Batch 200/718 | Loss: 1.6047
Epoch 2 | Batch 300/718 | Loss: 1.2780
Epoch 2 | Batch 400/718 | Loss: 1.4813
Epoch 2 | Batch 500/718 | Loss: 1.5728
Epoch 2 | Batch 600/718 | Loss: 1.4117
Epoch 2 | Batch 700/718 | Loss: 1.3565
Epoch 2, Loss: 1.391, Train Acc: 0.465 Val Acc: 0.486
Epoch 3 | Batch 0/718 | Loss: 1.5664
Epoch 3 | Batch 100/718 | Loss: 1.0828
Epoch 3 | Batch 200/718 | Loss: 1.1632
Epoch 3 | Batch 300/718 | Loss: 1.2912
Epoch 3 | Batch 400/718 | Loss: 1.1597
Epoch 3 | Batch 500/718 | Loss: 1.3914
Epoch 3 | Batch 600/718 | Loss: 1.0845
E

In [12]:
torch.save(model.state_dict(), 'emotion_model_v3.pth')